In [1]:
# Oyunlar Veri Temizleme Notebook'u
import pandas as pd
import numpy as np
import os
import re

In [2]:
# Veri setlerini okuyalım
df1 = pd.read_csv("../../data/raw/appstore.csv")
df2 = pd.read_csv("../../data/raw/playstore-tablet.csv")
df3 = pd.read_csv("../../data/raw/playstore-telefon.csv")
df4 = pd.read_csv("../../data/raw/playstore-windows.csv")
df5 = pd.read_csv("../../data/raw/steam-epicgames-nintendo-playstation.csv")

# Veri setlerini listeye alalım
dataframes = {
    "Appstore": df1,
    "Playstore Tablet": df2,
    "Playstore Telefon": df3,
    "Playstore Windows": df4,
    "Steam/Epic/etc": df5
}

# Veri setlerinin özet bilgilerini yazdıralım
for name, df in dataframes.items():
    print(f"--- {name} ---")
    print(f"Boyut: {df.shape}")
    print(f"Kolonlar: {df.columns.tolist()}")
    print("\n")

--- Appstore ---
Boyut: (398, 6)
Kolonlar: ['URL', 'Puan', 'Yaş Derecelendirmesi', 'Kategori', 'Geliştirici', 'Dil']


--- Playstore Tablet ---
Boyut: (156, 7)
Kolonlar: ['oyun_adi', 'puan', 'fiyat', 'yas_siniri', 'guncellenme_tarihi', 'tur_adi', 'URL']


--- Playstore Telefon ---
Boyut: (360, 8)
Kolonlar: ['oyun_adi', 'puan', 'fiyat', 'yas_siniri', 'guncellenme_tarihi', 'indirilme_sayisi', 'tur_adi', 'URL']


--- Playstore Windows ---
Boyut: (989, 8)
Kolonlar: ['oyun_adi', 'puan', 'fiyat', 'yas_siniri', 'guncellenme_tarihi', 'indirilme_sayisi', 'tur_adi', 'URL']


--- Steam/Epic/etc ---
Boyut: (19936, 11)
Kolonlar: ['oyun_adi', 'oyun_turu', 'gelistirici', 'puan', 'turkce_destegi', 'fiyat', 'yas', 'dil', 'ingilizce_destegi', 'dil_sayisi', 'platform']




In [3]:
# ---------------------------------------------------------
# 1. Appstore (df1) İşlemleri
# ---------------------------------------------------------
# Sileceğimiz sütunlar: URL, Dil
df1_clean = df1.drop(columns=['Dil', 'Puan'], errors='ignore')

# İsimleri standartlaştırma
df1_clean = df1_clean.rename(columns={
    'Yaş Derecelendirmesi': 'yas_siniri',
    'Kategori': 'kategori',
    'Geliştirici': 'gelistirici',
    'URL': 'oyun_adi'
})

# Eksik master sütunları ekleme
# Not: Oyun adını daha sonra çekeceğini belirttiğin için şimdilik boş (NaN) bırakıyoruz.
df1_clean['platform'] = 'Appstore'
df1_clean['cihaz_turu'] = 'Telefon'
df1_clean['fiyat'] = np.nan

# -------------------------------------------------------

guncel_kur = 44.52

# Playstore fiyatlarını hem temizleyen hem de anında Dolara (USD) çeviren fonksiyon
def tl_to_usd_cevir(deger):
    if pd.isna(deger):
        return np.nan
    
    # Virgüllü TL kuruşlarını noktaya çevir ve metni küçült
    deger = str(deger).lower().strip().replace(',', '.')
    
    # Ücretsiz oyunları 0.0 yap (Kur bölmesine girmesin)
    if 'free' in deger or 'ücretsiz' in deger or deger == '0' or deger == '0.0':
        return 0.0
    
    # İçindeki TL, ₺ gibi harf ve işaretleri silip sadece sayıyı al
    sadece_rakam = re.sub(r'[^\d.]', '', deger)
    
    try:
        # TL değerini kura bölerek USD'ye dönüştür
        return float(sadece_rakam) / guncel_kur
    except:
        return np.nan

# ---------------------------------------------------------
# 2. Playstore Tablet (df2) İşlemleri
# ---------------------------------------------------------
df2_clean = df2.drop(columns=['URL', 'guncellenme_tarihi', 'puan'], errors='ignore')
df2_clean = df2_clean.rename(columns={'tur_adi': 'kategori'})

df2_clean['platform'] = 'Playstore'
df2_clean['cihaz_turu'] = 'Tablet'
df2_clean['gelistirici'] = np.nan

# Fiyatları Dolara (USD) sabitleme
df2_clean['fiyat'] = df2_clean['fiyat'].apply(tl_to_usd_cevir)

# ---------------------------------------------------------
# 3. Playstore Telefon (df3) İşlemleri
# ---------------------------------------------------------
df3_clean = df3.drop(columns=['URL', 'guncellenme_tarihi', 'puan', 'indirilme_sayisi'], errors='ignore')
df3_clean = df3_clean.rename(columns={'tur_adi': 'kategori'})

df3_clean['platform'] = 'Playstore'
df3_clean['cihaz_turu'] = 'Telefon'
df3_clean['gelistirici'] = np.nan

# Fiyatları Dolara (USD) sabitleme
df3_clean['fiyat'] = df3_clean['fiyat'].apply(tl_to_usd_cevir)

# ---------------------------------------------------------
# 4. Playstore Windows (df4) İşlemleri
# ---------------------------------------------------------
df4_clean = df4.drop(columns=['URL', 'guncellenme_tarihi', 'puan', 'indirilme_sayisi'], errors='ignore')
df4_clean = df4_clean.rename(columns={'tur_adi': 'kategori'})

df4_clean['platform'] = 'Playstore'
df4_clean['cihaz_turu'] = 'PC'
df4_clean['gelistirici'] = np.nan

# Fiyatları Dolara (USD) sabitleme
df4_clean['fiyat'] = df4_clean['fiyat'].apply(tl_to_usd_cevir)

# ---------------------------------------------------------
# 5. Steam/Epic/etc (df5) İşlemleri (Güncellenmiş)
# ---------------------------------------------------------
df5_clean = df5.drop(columns=['turkce_destegi', 'dil', 'ingilizce_destegi', 'dil_sayisi', 'puan'], errors='ignore')
df5_clean = df5_clean.rename(columns={
    'oyun_turu': 'kategori',
    'yas': 'yas_siniri'
})

# --- Dinamik Cihaz Türü Ataması ---
pc_platformlari = {"Epic Games", "Steam"}
konsol_platformlari = {"Nintendo", "PlayStation"}

def cihaz_turu_etiketi(platform_serisi):
    platformlar = set(platform_serisi.dropna().unique())
    pc_var = bool(platformlar & pc_platformlari)
    konsol_var = bool(platformlar & konsol_platformlari)

    if pc_var and konsol_var:
        return "PC, Konsol"
    if pc_var:
        return "PC"
    if konsol_var:
        return "Konsol"
    return np.nan

# oyun_adi bazında gruplayarak cihaz türünü atama
df5_clean["cihaz_turu"] = (
    df5_clean.groupby("oyun_adi", dropna=False)["platform"]
      .transform(cihaz_turu_etiketi)
)

# ---------------------------------------------------------
# 6. DataFrame Birleştirme (Concatenation)
# ---------------------------------------------------------
# İstediğimiz altın sütun sırası
hedef_sutunlar = [
    'oyun_adi', 'platform', 'cihaz_turu', 'kategori', 
    'yas_siniri', 'fiyat', 'gelistirici'
]

# Tüm temizlenmiş df'leri birleştirme
df_oyunlar = pd.concat([df1_clean, df2_clean, df3_clean, df4_clean, df5_clean], ignore_index=True)

# Sütun sırasını tam istediğimiz vizyona göre hizalama
df_oyunlar = df_oyunlar[hedef_sutunlar]

# İşlem sonucunu kontrol etme
print("Master DataFrame Boyutu:", df_oyunlar.shape)
print("\nSütunlar ve Veri Tipleri:\n", df_oyunlar.dtypes)
display(df_oyunlar.head())

Master DataFrame Boyutu: (21839, 7)

Sütunlar ve Veri Tipleri:
 oyun_adi        object
platform        object
cihaz_turu      object
kategori        object
yas_siniri      object
fiyat          float64
gelistirici     object
dtype: object


,oyun_adi,platform,cihaz_turu,kategori,yas_siniri,fiyat,gelistirici
0,https://apps.apple.com/tr/app/block-blast/id16...,Appstore,Telefon,Bulmaca,13+,NaN,Hungry Studio
1,https://apps.apple.com/tr/app/super-bear-adven...,Appstore,Telefon,Aksiyon,4+,NaN,Earthkwak Games
2,https://apps.apple.com/tr/app/brawl-stars/id12...,Appstore,Telefon,Aksiyon,13+,NaN,Supercell
3,https://apps.apple.com/tr/app/arrows-puzzle-es...,Appstore,Telefon,Bulmaca,9+,NaN,Lessmore GmbH
4,https://apps.apple.com/tr/app/kingshot/id67395...,Appstore,Telefon,Strateji,9+,NaN,Century Games Pte. Ltd.


In [4]:
df_oyunlar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21839 entries, 0 to 21838
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   oyun_adi     21830 non-null  object 
 1   platform     21839 non-null  object 
 2   cihaz_turu   21839 non-null  object 
 3   kategori     21547 non-null  object 
 4   yas_siniri   20774 non-null  object 
 5   fiyat        19986 non-null  float64
 6   gelistirici  13282 non-null  object 
dtypes: float64(1), object(6)
memory usage: 1.2+ MB


In [5]:
print("="*60)
print("🛠️ KUSURSUZ YAŞ SINIRI TEMİZLİĞİ (NOKTA HATASI ÇÖZÜLDÜ)")
print("="*60)

# 1. str.extract(r'(\d+)') ile sadece ilk rakam grubunu (tam sayıyı) çekiyoruz!
# '18.0' -> '18' olur. | '13+' -> '13' olur.
df_oyunlar['yas_siniri'] = df_oyunlar['yas_siniri'].astype(str).str.extract(r'(\d+)')

# 2. Sayısal değere dönüştürüyoruz
df_oyunlar['yas_siniri'] = pd.to_numeric(df_oyunlar['yas_siniri'], errors='coerce')

# 3. Boş (NaN) kalanları 0 (Tüm Yaşlar) kabul edip, tam sayıya kilitliyoruz
df_oyunlar['yas_siniri'] = df_oyunlar['yas_siniri'].fillna(0).astype(int)

# 4. Daha önce yaptığımız "PEGI Standartlaştırma" işlemini de buraya ekleyelim ki her şey tek kalemde bitsin
yas_duzeltme_sozlugu = {
    4: 3,
    6: 7,
    9: 10,
    13: 12,
    15: 16
}
df_oyunlar['yas_siniri'] = df_oyunlar['yas_siniri'].replace(yas_duzeltme_sozlugu)

# Yeni ve kusursuz dağılımı görelim!
print("✅ Temizlik başarılı! Yeni Yaş Dağılımı:")
print("-" * 50)
print(df_oyunlar['yas_siniri'].value_counts().sort_index())

🛠️ KUSURSUZ YAŞ SINIRI TEMİZLİĞİ (NOKTA HATASI ÇÖZÜLDÜ)
✅ Temizlik başarılı! Yeni Yaş Dağılımı:
--------------------------------------------------
yas_siniri
0     4482
3     5907
7     2379
10    1042
12    4112
16    2154
18    1763
Name: count, dtype: int64


In [6]:
# Eksik verilerin hem sayısını hem de yüzdesini hesaplıyoruz
eksik_yuzde = (df_oyunlar.isnull().sum() / len(df_oyunlar)) * 100

# Şık bir tabloya dönüştürüp, en çok eksiği olandan en aza doğru sıralıyoruz
eksik_tablosu = pd.DataFrame({
    'Eksik_Değer_Sayısı': df_oyunlar.isnull().sum(),
    'Eksik_Oranı (%)': eksik_yuzde.round(2)
}).sort_values(by='Eksik_Oranı (%)', ascending=False)

print("📌 Sütun Bazlı Eksik Veri Analizi:\n")
display(eksik_tablosu)

📌 Sütun Bazlı Eksik Veri Analizi:



,Eksik_Değer_Sayısı,Eksik_Oranı (%)
gelistirici,8557,39.18
fiyat,1853,8.48
kategori,292,1.34
oyun_adi,9,0.04
platform,0,0.00
cihaz_turu,0,0.00
yas_siniri,0,0.00


In [7]:
print(f"İşlem öncesi veri boyutu: {df_oyunlar.shape}")

# ---------------------------------------------------------
# 1. ADIM: Fiyat Sütununu Sayısallaştırma (Medyan alabilmek için şart)
# ---------------------------------------------------------
def fiyat_temizle(deger):
    if pd.isna(deger):
        return np.nan
    deger = str(deger).lower().strip()
    if 'free' in deger or 'ücretsiz' in deger or deger == '0' or deger == '0.0':
        return 0.0
    sadece_rakam = re.sub(r'[^\d.]', '', deger)
    try:
        return float(sadece_rakam)
    except:
        return np.nan

df_oyunlar['fiyat'] = df_oyunlar['fiyat'].apply(fiyat_temizle)

# ---------------------------------------------------------
# 2. ADIM: Kritik Eksiklerin Silinmesi (Dropna)
# ---------------------------------------------------------
df_oyunlar.dropna(subset=['oyun_adi', 'kategori', 'yas_siniri'], inplace=True)

# ---------------------------------------------------------
# 3. ADIM: Geliştirici Sütununa "Bilinmiyor" Ataması
# ---------------------------------------------------------
df_oyunlar['gelistirici'] = df_oyunlar['gelistirici'].fillna('Bilinmiyor')

# Sonucu kontrol edelim
print(f"İşlem sonrası veri boyutu: {df_oyunlar.shape}")
print("\nKalan Eksik Veriler:")
print(df_oyunlar.isnull().sum())

İşlem öncesi veri boyutu: (21839, 7)
İşlem sonrası veri boyutu: (21542, 7)

Kalan Eksik Veriler:
oyun_adi          0
platform          0
cihaz_turu        0
kategori          0
yas_siniri        0
fiyat          1827
gelistirici       0
dtype: int64


In [8]:
# 1. Birebir Aynı Olan Satırlar (Tüm sütunları tıpatıp aynı)
# Bu genellikle kazıma (scraping) sırasında aynı sayfanın iki kez çekilmesinden kaynaklanır.
birebir_tekrar = df_oyunlar.duplicated().sum()

# 2. Sadece Oyun Adı Aynı Olan Satırlar 
# Bu durum çok normaldir çünkü aynı oyun hem Appstore'da hem Playstore'da hem de PC'de olabilir.
oyun_adi_tekrar = df_oyunlar.duplicated(subset=['oyun_adi']).sum()

print("="*50)
print("🔍 DUPLİKE (TEKRAR EDEN) VERİ ANALİZİ")
print("="*50)
print(f"Birebir Aynı (Tamamen Duplike) Satır Sayısı: {birebir_tekrar}")
print(f"Sadece Oyun Adı Aynı Olan (Farklı Platform/Cihaz) Satır Sayısı: {oyun_adi_tekrar}")

🔍 DUPLİKE (TEKRAR EDEN) VERİ ANALİZİ
Birebir Aynı (Tamamen Duplike) Satır Sayısı: 240
Sadece Oyun Adı Aynı Olan (Farklı Platform/Cihaz) Satır Sayısı: 3357


In [9]:
print(f"Temizlik öncesi satır sayısı: {df_oyunlar.shape[0]}")

# Sadece birebir aynı olan çöp satırları siliyoruz
df_oyunlar = df_oyunlar.drop_duplicates(keep='first')

# İndeksi sıfırlıyoruz ki satır numaraları pürüzsüz ilerlesin
df_oyunlar = df_oyunlar.reset_index(drop=True)

print(f"Temizlik sonrası satır sayısı: {df_oyunlar.shape[0]}")
print("✅ Birebir tekrarlar silindi, platform farklılıkları (farklı versiyonlar) başarıyla korundu.")

Temizlik öncesi satır sayısı: 21542
Temizlik sonrası satır sayısı: 21302
✅ Birebir tekrarlar silindi, platform farklılıkları (farklı versiyonlar) başarıyla korundu.


In [10]:
import pandas as pd

# Konsolu boğmamak için es geçeceğimiz yüksek kardinaliteli sütunlar
es_gecilecekler = ['oyun_adi', 'gelistirici', 'kategori']

print("="*60)
print("🎯 HEDEF SÜTUNLARIN EKSİKSİZ (TAM) VALUE COUNTS İNCELEMESİ")
print("="*60)

for col in df_oyunlar.columns:
    # Eğer sütun es geçilecekler listesindeyse döngüyü atla
    if col in es_gecilecekler:
        continue
    
    # Dropna=False ile NaN değerleri de eşsiz değer sayısına dahil ediyoruz
    print(f"\n📌 SÜTUN: '{col}' (Toplam Eşsiz Değer: {df_oyunlar[col].nunique(dropna=False)})")
    print("-" * 50)
    
    # Sadece bu döngü adımı için Pandas'ın satır gösterme limitini (max_rows) sonsuz (None) yapıyoruz
    with pd.option_context('display.max_rows', None):
        print(df_oyunlar[col].value_counts(dropna=False))
        
    print("-" * 50)

🎯 HEDEF SÜTUNLARIN EKSİKSİZ (TAM) VALUE COUNTS İNCELEMESİ

📌 SÜTUN: 'platform' (Toplam Eşsiz Değer: 6)
--------------------------------------------------
platform
PlayStation    6674
Epic Games     5276
Steam          4076
Nintendo       3402
Playstore      1489
Appstore        385
Name: count, dtype: int64
--------------------------------------------------

📌 SÜTUN: 'cihaz_turu' (Toplam Eşsiz Değer: 5)
--------------------------------------------------
cihaz_turu
PC            8319
Konsol        7941
PC, Konsol    4142
Telefon        744
Tablet         156
Name: count, dtype: int64
--------------------------------------------------

📌 SÜTUN: 'yas_siniri' (Toplam Eşsiz Değer: 7)
--------------------------------------------------
yas_siniri
3     5841
0     4383
12    3969
7     2358
16    2100
18    1641
10    1010
Name: count, dtype: int64
--------------------------------------------------

📌 SÜTUN: 'fiyat' (Toplam Eşsiz Değer: 1230)
--------------------------------------------------


In [11]:
print("="*50)
print("🧹 YAŞ SINIRI STANDARTLAŞTIRMA İŞLEMİ")
print("="*50)

def yas_temizle(deger):
    if pd.isna(deger):
        return np.nan
    
    # Veriyi garanti olması için string'e çevir ve boşlukları al
    deger = str(deger).strip()
    
    # '+' veya harf gibi rakam ve nokta dışındaki tüm karakterleri sil
    sadece_rakam = re.sub(r'[^\d.]', '', deger)
    
    try:
        # Önce float'a çevirip 0.0 veya 15.0 gibi değerleri düzeltiyoruz
        # Sonra int() ile saf bir tam sayıya sabitliyoruz (Örn: 4.0 -> 4)
        return int(float(sadece_rakam))
    except:
        return np.nan

# Temizleme fonksiyonunu sütuna uyguluyoruz
df_oyunlar['yas_siniri'] = df_oyunlar['yas_siniri'].apply(yas_temizle)

# İşlem sonucunu kontrol ediyoruz
print("✅ Yaş sınırı temizliği tamamlandı!\n")
print(f"📌 SÜTUN: 'yas_siniri' (Yeni Toplam Eşsiz Değer: {df_oyunlar['yas_siniri'].nunique()})")
print("-" * 30)
print(df_oyunlar['yas_siniri'].value_counts(dropna=False).sort_index())
print("-" * 30)

🧹 YAŞ SINIRI STANDARTLAŞTIRMA İŞLEMİ
✅ Yaş sınırı temizliği tamamlandı!

📌 SÜTUN: 'yas_siniri' (Yeni Toplam Eşsiz Değer: 7)
------------------------------
yas_siniri
0     4383
3     5841
7     2358
10    1010
12    3969
16    2100
18    1641
Name: count, dtype: int64
------------------------------


In [12]:
print("="*60)
print("🎯 KATEGORİ SÜTUNU: PARÇALANMIŞ (EXPLODE) TAM LİSTE")
print("="*60)

# 1. Eksik verileri (NaN) bu analizlik es geçip tüm satırları metne (string) çeviriyoruz
# 2. split(',') ile "Aksiyon, Macera" gibi verileri virgül noktasından listeye dönüştürüyoruz
kategoriler_listesi = df_oyunlar['kategori'].dropna().astype(str).str.split(',')

# 3. explode() ile o listenin içindeki her bir elemanı alt alta yepyeni satırlara ayırıyoruz
# 4. str.strip() ile başındaki veya sonundaki gizli boşlukları siliyoruz (örn: " Macera" -> "Macera")
tekil_kategoriler = kategoriler_listesi.explode().str.strip()

# Artık elimizde saf ve tekil kategorilerin oluşturduğu o devasa havuz var.
# Şimdi bu havuzun röntgenini çekebiliriz!
kategori_frekanslari = tekil_kategoriler.value_counts()

# Tüm listeyi hiçbir kesinti olmadan ekrana döküyoruz
with pd.option_context('display.max_rows', None):
    print(kategori_frekanslari)

🎯 KATEGORİ SÜTUNU: PARÇALANMIŞ (EXPLODE) TAM LİSTE
kategori
Tek Oyunculu                          7049
Aksiyon                               5803
Macera                                5070
Bağımsız                              3363
Simülasyon                            3306
Bulmaca                               2685
Strateji                              2503
Basit Eğlence                         2310
Action                                1892
Kumanda Desteği                       1821
RYO                                   1698
Nişancı                               1549
Arcade                                1498
Bulut Kayıtları                       1418
Korku                                 1282
Adventure                             1250
Eşli                                  1189
Birinci Şahıs                         1152
Atmosferik                            1096
Keşif                                 1093
2D                                     999
Günlük                               

In [13]:
len(kategori_frekanslari)

625

In [14]:
df_oyunlar.sample(10)

,oyun_adi,platform,cihaz_turu,kategori,yas_siniri,fiyat,gelistirici
15748,Goosebumps: Terror in Little Creek,PlayStation,Konsol,Korku,12,39.29,Bilinmiyor
16563,Barista Simulator,PlayStation,Konsol,Simülasyon,12,15.70,Bilinmiyor
11363,Tactics Ogre: Reborn,Nintendo,Konsol,"Role-Playing, Strategy",12,49.99,Square Enix
2564,Night Swarm,Steam,"PC, Konsol","Aksiyon Roguelike, Vampir, Mermi Cehennemi, Ak...",12,3.49,Fubu Games
7994,Hero of the Kingdom,Epic Games,PC,"RYO, Bulut Kayıtları, Tek Oyunculu, Macera, Ba...",12,2.49,Lonely Troops
133,https://apps.apple.com/tr/app/petrolhead-extre...,Appstore,Telefon,Yarış,3,NaN,Lethe Studios
7926,Escape From Mystwood Mansion,Epic Games,"PC, Konsol","Kumanda Desteği, Macera, Bulut Kayıtları, Tek ...",3,5.26,Lost Sock Studio
20135,Family Mysteries 3: Criminal Mindset,PlayStation,Konsol,"Macera, Günlük, Bulmaca",16,NaN,Bilinmiyor
11904,Wandersong,Nintendo,Konsol,"Action, Adventure",10,19.99,Greg Lobanov
16386,Tema Parkı Reçeli,PlayStation,Konsol,Bulmaca,3,1.55,Bilinmiyor


In [15]:
print("="*60)
print("🧹 YAŞ SINIRI STANDARTLAŞTIRMA (PEGI FORMATINA ÇEKME)")
print("="*60)

# Değiştirilecek ara yaş sınırları için bir sözlük (dictionary) oluşturuyoruz
yas_duzeltme_sozlugu = {
    4: 3,
    6: 7,
    9: 10,
    13: 12,
    15: 16
}

# .replace() fonksiyonu ile sadece sözlükteki değerleri güncelliyoruz.
# Sözlükte olmayan ana PEGI değerleri (0, 3, 7, 10, 12, 16, 18) bozulmadan kalır.
df_oyunlar['yas_siniri'] = df_oyunlar['yas_siniri'].replace(yas_duzeltme_sozlugu)

# İşlemin başarılı olduğunu görmek için yeni dağılımı kontrol edelim
print("✅ Yaş sınırları başarıyla standartlaştırıldı!\nGüncel Yaş Dağılımı:")
print(df_oyunlar['yas_siniri'].value_counts().sort_index())

🧹 YAŞ SINIRI STANDARTLAŞTIRMA (PEGI FORMATINA ÇEKME)
✅ Yaş sınırları başarıyla standartlaştırıldı!
Güncel Yaş Dağılımı:
yas_siniri
0     4383
3     5841
7     2358
10    1010
12    3969
16    2100
18    1641
Name: count, dtype: int64


In [16]:
print("="*60)
print("🚨 TEHLİKE (RİSK) SÜTUNU OLUŞTURMA İŞLEMİ")
print("="*60)

# 1. Interim klasöründeki sözlüğümüzü okuyoruz
dosya_yolu = "../../data/interim/guvenli-tehlikeli.csv"
df_sozluk = pd.read_csv(dosya_yolu)

# 2. Sözlükteki boşlukları temizleme (Kritik Adım)
# Hem kategori adlarındaki hem de etiketlerdeki sağ/sol boşlukları siliyoruz
df_sozluk['kategori'] = df_sozluk['kategori'].str.strip()
# Sütun adın tam olarak 'Guvenli/Tehlikeli' görünüyor
df_sozluk['Guvenli/Tehlikeli'] = df_sozluk['Guvenli/Tehlikeli'].str.strip() 

# 3. Sadece 'Tehlikeli' olan kategorileri hızlı arama için bir Kümeye (Set) alıyoruz
# Set kullanmak, Python'un arama hızını listelere göre 100 kat artırır
tehlikeli_set = set(df_sozluk[df_sozluk['Guvenli/Tehlikeli'] == 'Tehlikeli']['kategori'])
print(f"Sözlükten çekilen toplam Tehlikeli Kategori Sayısı: {len(tehlikeli_set)}")

# 4. Ana verimiz için Tehlike Tespit Fonksiyonu
def tehlike_skoru_hesapla(kategori_metni):
    # Eğer kategori boşsa (NaN) güvenli (0) say
    if pd.isna(kategori_metni):
        return 0
    
    # "Action, Adventure, Role-Playing" gibi metni virgülden bölüp listeye çeviriyoruz
    # Ve her birinin etrafındaki boşlukları siliyoruz
    oyun_kategorileri = [k.strip() for k in str(kategori_metni).split(',')]
    
    # Oyunun kategorilerinden HERHANGİ BİRİ tehlikeli setimizde var mı?
    for k in oyun_kategorileri:
        if k in tehlikeli_set:
            return 1 # Eşleşme bulundu, oyunu tehlikeli (1) işaretle ve döngüyü bitir
            
    # Hiçbir eşleşme bulunamadıysa güvenli (0) işaretle
    return 0

# 5. Fonksiyonu DataFrame'e uyguluyoruz
df_oyunlar['tehlikeli_kategorisi'] = df_oyunlar['kategori'].apply(tehlike_skoru_hesapla)

# 6. İşlem Sonucunu Kontrol Edelim
print("\n✅ İşlem Başarılı! Yeni sütun dağılımı:")
print(df_oyunlar['tehlikeli_kategorisi'].value_counts(dropna=False))

🚨 TEHLİKE (RİSK) SÜTUNU OLUŞTURMA İŞLEMİ
Sözlükten çekilen toplam Tehlikeli Kategori Sayısı: 100

✅ İşlem Başarılı! Yeni sütun dağılımı:
tehlikeli_kategorisi
0    16122
1     5180
Name: count, dtype: int64


In [17]:
df_oyunlar["yas_siniri"].value_counts(dropna=False).sort_index()

yas_siniri
0     4383
3     5841
7     2358
10    1010
12    3969
16    2100
18    1641
Name: count, dtype: int64

In [18]:
import pandas as pd

print("="*60)
print("🚨 GÜNCELLENMİŞ YAŞ-KATEGORİ REGÜLASYON MOTORU")
print("="*60)

# 1. Kural Sözlüğünün Hazırlanması (Dokunulmadı, mantık aynı)
yas_kurallari = {
    "0_3": set([k.strip().lower() for k in ["Nişancı", "Korku", "Rekabetçi", "Zor", "Çatışma", "Karanlık", "FPS", "Psikolojik Korku", "Gizem", "Dövüş", "Şiddet", "Vahşet", "Mermi Cehennemi", "Hayatta Kalma - Korku", "Karanlık Fantezi", "Savaş", "Kıyamet Sonrası", "TPS", "Askerî", "Kara Mizah", "Cinsel İçerik", "Sıra Tabanlı Savaş", "Psikolojik", "Shoot 'Em Up", "Doğaüstü", "Zombi", "Suç", "Çıplaklık", "Gerilim", "Kartlı Savaş", "Üstten Görünüşlü Nişancı", "Arena Nişancısı", "Yetişkinler İçin", "Yetişkin", "Romantik", "3D Dövüş", "Platform (Zorlu)", "İblisler", "Savaş Oyunu", "Silah Kişiselleştirme", "Çift Analoglu Nişancı", "Battle Royale", "Soygun", "Hentai", "Savaşlı Yarış", "İlişki Simülatörü", "Kalıcı Ölüm", "Yıkım", "2D Dövüş", "Lovecraft Temalı", "NSFW", "Suikastçı", "Kahraman Merkezli Nişancı", "2. Dünya Savaşı", "Cüce", "Araçlı Savaş", "Kara Komedi", "Boomer Shooter", "Kumar", "Taktiğe dayalı nişancı", "Hiciv", "Kılıçlı Dövüş", "Gotik", "Kan", "Kötü Ana Karakter", "Gizemli Zindan", "Vampir", "Hayatta kalma ve korku", "Görkemli Dövüş", "Dövüş Sanatları", "Keskin Nişancı", "Soğuk Savaş", "Salgın Simülatörü", "1. Dünya Savaşı", "Tank", "On-Rails Shooter", "Şehir kurma ve savaş", "Acımasız", "Bağımlılık Yapıcı", "Dinozor", "Warhammer 40K", "Deniz Savaşı", "Araçlı savaş", "Balon nişancısı", "Kumarhane", "Keskin nişancı", "Asimetrik savaş alanı", "Savaş oyunu", "Horror", "Kart savaşı", "Ani Korku", "Kahraman nişancı", "Flört", "Koşucu nişancı", "Kavga", "Extraction Shooter", "Fighting", "Shooter", "LGBTİ+", "Looter Shooter"]]),
    "4_7": set([k.strip().lower() for k in ["Nişancı", "Korku", "Dövüş", "Çatışma", "Savaş", "FPS", "TPS", "Şiddet", "Vahşet", "Zombi", "Karanlık", "Psikolojik", "Hayatta Kalma", "Battle Royale", "Kumar", "Cinsel İçerik", "NSFW", "Hentai", "Kan", "Acımasız", "İblisler", "Vampir", "Suikastçı", "Soygun", "Karanlık Fantezi", "Gerilim", "Extraction Shooter", "Looter Shooter", "Horror", "Ani Korku", "Psikolojik Korku"]]),
    "9_10": set([k.strip().lower() for k in ["Nişancı", "FPS", "TPS", "Dövüş", "Savaş", "Psikolojik Korku", "Vahşet", "Kan", "Battle Royale", "Kumar", "Cinsel İçerik", "NSFW", "Hentai", "Hayatta Kalma-Korku", "Karanlık Fantezi", "İblisler", "Suikastçı", "Soygun", "Acımasız", "Horror"]]),
    "12": set([k.strip().lower() for k in ["Psikolojik Korku", "Vahşet", "Kan", "Kumar", "Cinsel İçerik", "NSFW", "Hentai", "Hayatta Kalma-Korku (yoğun)", "Karanlık Fantezi (yoğun)", "İblisler", "Suikastçı", "Soygun", "Acımasız", "Looter Shooter", "Extraction Shooter", "Yetişkin", "Yetişkinler İçin"]]),
    "13_15": set([k.strip().lower() for k in ["Kumar", "Cinsel İçerik", "NSFW", "Hentai", "Vahşet (aşırı)", "Kan (grafik)", "Acımasız (aşırı)", "Yetişkinler İçin"]]),
    "16": set([k.strip().lower() for k in ["Kumar (gerçek para)", "Cinsel İçerik (yoğun)", "NSFW", "Hentai", "Vahşet (aşırı)", "Yetişkin", "Yetişkinler İçin"]]),
    "18": set()
}

# 2. Satır Satır Çalışacak Kural Algoritması (GÜNCELLENDİ)
def yas_kategori_denetleyici(row):
    yas = row['yas_siniri']
    kategori_metni = row['kategori']

    if pd.isna(yas) or pd.isna(kategori_metni):
        return 0

    # YAŞ STANDARTLAŞTIRMASI SONRASI NOKTA ATIŞI (EXACT MATCH) KONTROLLER:
    if yas in [0, 3]:
        hedef_kume = yas_kurallari["0_3"]
    elif yas == 7:  # Eskiden <=7 idi, artık 4 ve 6 kalmadığı için doğrudan 7
        hedef_kume = yas_kurallari["4_7"]
    elif yas == 10: # Eskiden <=10 idi, 9 kalmadığı için doğrudan 10
        hedef_kume = yas_kurallari["9_10"]
    elif yas == 12:
        hedef_kume = yas_kurallari["12"]
    elif yas in [13, 14, 15]: 
        # 13 ve 15'i standartlaştırdın ama veride gözden kaçan bir 14 kalmış olma 
        # ihtimaline karşı bu kural bloğunu bir "güvenlik ağı" olarak tutuyoruz.
        hedef_kume = yas_kurallari["13_15"]
    elif yas == 16:
        hedef_kume = yas_kurallari["16"]
    else: 
        return 0 # 18 ve üstü (veya tanımsız dev sayılar) için güvenli kabul et

    # Oyunun kategorilerini parçalayıp, küçük harfe çevirip, boşluklarını siliyoruz
    oyun_kategorileri = [k.strip().lower() for k in str(kategori_metni).split(',')]

    # Yasaklı kelime taraması
    for k in oyun_kategorileri:
        if k in hedef_kume:
            return 1 # Yakalandı! Tehlikeli olarak işaretle.

    return 0 

# 3. Kural Motorunu DataFrame'e Uygulama
df_oyunlar['tehlikeli_oyun'] = df_oyunlar.apply(yas_kategori_denetleyici, axis=1)

# İşlem Sonucunu Raporlayalım
print("✅ Yaş ve Kategori denetimi güncel standartlara göre tamamlandı!\n")
print("Tehlikeli Oyun Dağılımı:")
print(df_oyunlar['tehlikeli_oyun'].value_counts())

🚨 GÜNCELLENMİŞ YAŞ-KATEGORİ REGÜLASYON MOTORU
✅ Yaş ve Kategori denetimi güncel standartlara göre tamamlandı!

Tehlikeli Oyun Dağılımı:
tehlikeli_oyun
0    18336
1     2966
Name: count, dtype: int64


In [19]:
df_oyunlar.sample(10, random_state=42)

,oyun_adi,platform,cihaz_turu,kategori,yas_siniri,fiyat,gelistirici,tehlikeli_kategorisi,tehlikeli_oyun
7515,The Serpent Rogue,Epic Games,"PC, Konsol","RYO, Tek Oyunculu, Aksiyon-Macera, Bağımsız",7,9.88,SENGI GAMES,0,0
13371,Avatar The Last Airbender: Quest for Balance,Nintendo,Konsol,"Action, Adventure",10,49.99,BamTang Games,0,0
16581,ARMORED CORE MASTER OF ARENA,PlayStation,Konsol,Aksiyon,12,NaN,Bilinmiyor,0,0
12268,Sonic Origins,Nintendo,"PC, Konsol",Action,3,29.99,SEGA,0,0
20415,Horizon Zero Dawn™ Remastered,PlayStation,"PC, Konsol","Rol Yapma Oyunları, Aksiyon",16,48.27,Bilinmiyor,0,0
18539,NeonPowerUp! PS4™ & PS5®,PlayStation,Konsol,"Aksiyon, Bulmaca, Aksiyon",3,1.57,Bilinmiyor,0,0
12622,The End Is Nigh,Nintendo,Konsol,Action,18,14.99,Edmund McMillen,0,0
856,Jewel Hunter - Match 3 Games,Playstore,Telefon,"Bulmaca, Üçlü Eşleme, Üçlü eşleştirme macerası...",3,0.00,Bilinmiyor,0,0
13801,Capes,Nintendo,"PC, Konsol","Action, Role-Playing, Strategy",12,39.99,Daedalic Entertainment,0,0
5910,FATAL FURY: City of the Wolves,Steam,PC,"2D Dövüş, Aksiyon, 2D, PvP, Görkemli Dövüş, Ço...",0,37.99,SNK CORPORATION,1,1


In [20]:
print("="*70)
print("🔍 'TEHLİKELİ' (Genel Sözlük) vs 'TEHLİKELİ_OYUN' (Yaş Kurallı) ANALİZİ")
print("="*70)

# 1. Şık ve Profesyonel Çapraz Tablo (Crosstab)
karsilastirma_tablosu = pd.crosstab(
    df_oyunlar['tehlikeli_kategorisi'], 
    df_oyunlar['tehlikeli_oyun'], 
    rownames=['Genel Sözlük (tehlikeli)'], 
    colnames=['Yaş Kurallı (tehlikeli_oyun)'],
    margins=True, # Satır ve sütun sonlarına "Toplam" ekler
    margins_name="Toplam"
)

display(karsilastirma_tablosu)

# 2. Rakamları Değişkenlere Atayıp Net Bir Şekilde Yazdırma
ikisinde_de_var = len(df_oyunlar[(df_oyunlar['tehlikeli_kategorisi'] == 1) & (df_oyunlar['tehlikeli_oyun'] == 1)])
sadece_genel_sozluk = len(df_oyunlar[(df_oyunlar['tehlikeli_kategorisi'] == 1) & (df_oyunlar['tehlikeli_oyun'] == 0)])
sadece_yas_kuralli = len(df_oyunlar[(df_oyunlar['tehlikeli_kategorisi'] == 0) & (df_oyunlar['tehlikeli_oyun'] == 1)])
ikisinde_de_yok = len(df_oyunlar[(df_oyunlar['tehlikeli_kategorisi'] == 0) & (df_oyunlar['tehlikeli_oyun'] == 0)])

print("\n📊 DETAYLI SAYIM DÖKÜMÜ:")
print("-" * 50)
print(f"🔴 İkisinde de Tehlikeli (1-1): {ikisinde_de_var} oyun")
print(f"   (İçerik kötü ve küçük yaşa sunulmuş - TAM İHLAL)")

print(f"\n🟡 Sadece Genel Sözlükte Tehlikeli (1-0): {sadece_genel_sozluk} oyun")
print(f"   (İçerik kötü ama yaşı büyük/uygun ayarlanmış - YASAL)")

print(f"\n🟠 Sadece Yaş Kuralında Tehlikeli (0-1): {sadece_yas_kuralli} oyun")
print(f"   (Genel listende olmayıp yaş kuralına takılan kaçaklar)")

print(f"\n🟢 İkisinde de Güvenli (0-0): {ikisinde_de_yok} oyun")
print(f"   (Hem içerik temiz hem yaşa uygun - TAM GÜVENLİ)")
print("-" * 50)

🔍 'TEHLİKELİ' (Genel Sözlük) vs 'TEHLİKELİ_OYUN' (Yaş Kurallı) ANALİZİ


Yaş Kurallı (tehlikeli_oyun),0,1,Toplam
Genel Sözlük (tehlikeli),,,
0,16100,22,16122
1,2236,2944,5180
Toplam,18336,2966,21302



📊 DETAYLI SAYIM DÖKÜMÜ:
--------------------------------------------------
🔴 İkisinde de Tehlikeli (1-1): 2944 oyun
   (İçerik kötü ve küçük yaşa sunulmuş - TAM İHLAL)

🟡 Sadece Genel Sözlükte Tehlikeli (1-0): 2236 oyun
   (İçerik kötü ama yaşı büyük/uygun ayarlanmış - YASAL)

🟠 Sadece Yaş Kuralında Tehlikeli (0-1): 22 oyun
   (Genel listende olmayıp yaş kuralına takılan kaçaklar)

🟢 İkisinde de Güvenli (0-0): 16100 oyun
   (Hem içerik temiz hem yaşa uygun - TAM GÜVENLİ)
--------------------------------------------------


In [21]:
import pandas as pd

# 1. GÜVENLİK ADIMI: Eğer sütun hala eski adındaysa (veya typo varsa), onu kesin olarak düzeltiyoruz
if 'tehlikeli' in df_oyunlar.columns:
    df_oyunlar = df_oyunlar.rename(columns={'tehlikeli': 'tehlikeli_kategori'})
elif 'tehlikeli_kategorisi' in df_oyunlar.columns:
    df_oyunlar = df_oyunlar.rename(columns={'tehlikeli_kategorisi': 'tehlikeli_kategori'})

# Hata ayıklama (Debugging) için mevcut sütunları ekrana basıp kendi gözlerimizle teyit ediyoruz
print("Mevcut Sütunlar:", df_oyunlar.columns.tolist())
print("-" * 50)

# 2. TABLO OLUŞTURMA ADIMI
yas_ozet_tablosu = df_oyunlar.groupby('yas_siniri').agg(
    Toplam_Oyun_Sayisi=('yas_siniri', 'size'),
    Tehlikeli_Kategori_Sayisi=('tehlikeli_kategori', 'sum'),
    Tehlikeli_Oyun_Sayisi=('tehlikeli_oyun', 'sum')
).reset_index()

# Tabloyu yaş sınırına göre küçükten büyüğe sıralıyoruz
yas_ozet_tablosu = yas_ozet_tablosu.sort_values(by='yas_siniri')

# Tüm satırların net görünmesi için ekrana basıyoruz
display(yas_ozet_tablosu)

Mevcut Sütunlar: ['oyun_adi', 'platform', 'cihaz_turu', 'kategori', 'yas_siniri', 'fiyat', 'gelistirici', 'tehlikeli_kategori', 'tehlikeli_oyun']
--------------------------------------------------


,yas_siniri,Toplam_Oyun_Sayisi,Tehlikeli_Kategori_Sayisi,Tehlikeli_Oyun_Sayisi
0,0,4383,2230,2230
1,3,5841,287,287
2,7,2358,407,352
3,10,1010,45,0
4,12,3969,811,47
5,16,2100,814,50
6,18,1641,586,0


In [22]:
df_oyunlar.sample(10, random_state=42)

,oyun_adi,platform,cihaz_turu,kategori,yas_siniri,fiyat,gelistirici,tehlikeli_kategori,tehlikeli_oyun
7515,The Serpent Rogue,Epic Games,"PC, Konsol","RYO, Tek Oyunculu, Aksiyon-Macera, Bağımsız",7,9.88,SENGI GAMES,0,0
13371,Avatar The Last Airbender: Quest for Balance,Nintendo,Konsol,"Action, Adventure",10,49.99,BamTang Games,0,0
16581,ARMORED CORE MASTER OF ARENA,PlayStation,Konsol,Aksiyon,12,NaN,Bilinmiyor,0,0
12268,Sonic Origins,Nintendo,"PC, Konsol",Action,3,29.99,SEGA,0,0
20415,Horizon Zero Dawn™ Remastered,PlayStation,"PC, Konsol","Rol Yapma Oyunları, Aksiyon",16,48.27,Bilinmiyor,0,0
18539,NeonPowerUp! PS4™ & PS5®,PlayStation,Konsol,"Aksiyon, Bulmaca, Aksiyon",3,1.57,Bilinmiyor,0,0
12622,The End Is Nigh,Nintendo,Konsol,Action,18,14.99,Edmund McMillen,0,0
856,Jewel Hunter - Match 3 Games,Playstore,Telefon,"Bulmaca, Üçlü Eşleme, Üçlü eşleştirme macerası...",3,0.00,Bilinmiyor,0,0
13801,Capes,Nintendo,"PC, Konsol","Action, Role-Playing, Strategy",12,39.99,Daedalic Entertainment,0,0
5910,FATAL FURY: City of the Wolves,Steam,PC,"2D Dövüş, Aksiyon, 2D, PvP, Görkemli Dövüş, Ço...",0,37.99,SNK CORPORATION,1,1


In [23]:
# 1. Hedef klasör yolunu belirleme 
# (notebooks/game klasöründen 2 tık yukarı çıkıp data/processed'e giriyoruz)
output_dir = "../../data/processed"

# Klasörün var olduğundan emin olalım (senin yapında var ama best-practice olarak eklenir)
os.makedirs(output_dir, exist_ok=True)

# 2. Dosya adını ve tam yolunu belirleme
# İsimlendirme standardına uyarak 'oyunlar_temiz.csv' yapıyoruz
output_path = os.path.join(output_dir, "oyunlar_temiz.csv")

# 3. DataFrame'i CSV olarak kaydetme
# index=False: Satır numaralarını gereksiz bir sütun olarak kaydetmemek için
# encoding='utf-8': Türkçe karakterlerin (ş, ğ, ç) bozulmaması için
df_oyunlar.to_csv(output_path, index=False, encoding='utf-8')

print(f"✅ Oyunlar Master veri seti başarıyla kaydedildi:\nKonum: {output_path}")
print(f"Veri Seti Boyutu: {df_oyunlar.shape}")

✅ Oyunlar Master veri seti başarıyla kaydedildi:
Konum: ../../data/processed\oyunlar_temiz.csv
Veri Seti Boyutu: (21302, 9)
